# Topic Classifier Dataset Pipeline
**Project:** SRH AI Platform — ALU Capstone  
**Model:** Model 3 — Multi-class SRH Topic Classifier  
**Purpose:** Download, clean, and merge four labeled datasets into one unified topic-labeled CSV.

## SRH Topic Classes (7 classes)
| ID | Label | Description |
|---|---|---|
| 0 | `contraception` | Birth control, family planning, condoms, pills, IUDs |
| 1 | `sti_hiv` | STIs, HIV/AIDS, STI testing, treatment, prevention |
| 2 | `pregnancy` | Pregnancy, antenatal care, childbirth, miscarriage, fertility |
| 3 | `puberty` | Puberty, menstruation, adolescent development, body changes |
| 4 | `gbv_consent` | Gender-based violence, consent, sexual coercion, abuse |
| 5 | `disability_srh` | SRH for persons with disabilities, inclusive sexual health |
| 6 | `general_srh` | General sexual health, relationships, hygiene, anatomy |

## Datasets Used
| Dataset | Source | Labeling method | Size |
|---|---|---|---|
| MedMCQA | openlifescienceai | Filter by `subject` column | ~194k (filter ~8k) |
| AdaptLLM med_knowledge_prob | AdaptLLM | Filename = topic class | ~16k total (filter ~3k) |
| HealthCareMagic-100k | lavita/Malikeh1375 | Keyword filter on question | ~112k (filter ~6k) |
| AfriMedQA v2 | intronhealth | Filter by `specialty` column | ~15k (filter ~2k) |

## Output
A single balanced CSV: `text`, `topic` (string), `label` (int 0–6), `source`

---
## STEP 0 — Install and Import

In [ ]:
!pip install -q datasets pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from collections import Counter

os.makedirs('data', exist_ok=True)

# ── Topic taxonomy ─────────────────────────────────────────────────────────────
TOPIC_MAP = {
    'contraception': 0,
    'sti_hiv':       1,
    'pregnancy':     2,
    'puberty':       3,
    'gbv_consent':   4,
    'disability_srh':5,
    'general_srh':   6,
}
TOPIC_NAMES = {v: k for k, v in TOPIC_MAP.items()}

# ── Keyword dictionaries for each topic (used in HealthCareMagic filtering) ────
TOPIC_KEYWORDS = {
    'contraception': [
        'contraception', 'contraceptive', 'birth control', 'family planning',
        'condom', 'pill', 'iud', 'implant', 'injection', 'depo', 'nexplanon',
        'morning after', 'plan b', 'emergency contraception', 'copper coil',
        'patch', 'diaphragm', 'sterilization', 'vasectomy', 'tubal ligation'
    ],
    'sti_hiv': [
        'hiv', 'aids', 'sti', 'std', 'sexually transmitted', 'chlamydia',
        'gonorrhea', 'gonorrhoea', 'syphilis', 'herpes', 'hpv', 'hepatitis b',
        'hepatitis c', 'trichomoniasis', 'bacterial vaginosis', 'discharge',
        'genital wart', 'antiretroviral', 'arvs', 'prep', 'pep', 'viral load',
        'cd4', 'std test', 'sti test'
    ],
    'pregnancy': [
        'pregnant', 'pregnancy', 'prenatal', 'antenatal', 'postnatal',
        'trimester', 'fetus', 'foetus', 'labour', 'labor', 'childbirth',
        'delivery', 'miscarriage', 'abortion', 'stillbirth', 'ectopic',
        'maternal', 'morning sickness', 'fertility', 'infertility',
        'ivf', 'ovulation', 'menstrual cycle', 'missed period', 'pregnancy test'
    ],
    'puberty': [
        'puberty', 'menstruation', 'menstrual', 'period', 'menarche',
        'adolescent', 'teenager', 'teen', 'pubescent', 'breast development',
        'body hair', 'acne', 'voice breaking', 'growth spurt', 'wet dream',
        'first period', 'irregular period', 'dysmenorrhea', 'pms', 'pmdd'
    ],
    'gbv_consent': [
        'sexual violence', 'rape', 'sexual assault', 'gbv', 'gender based violence',
        'gender-based violence', 'consent', 'coercion', 'forced sex', 'domestic violence',
        'intimate partner violence', 'sexual abuse', 'harassment', 'trafficking',
        'exploitation', 'femicide'
    ],
    'disability_srh': [
        'disability', 'disabled', 'wheelchair', 'visual impairment', 'hearing impairment',
        'deaf', 'blind', 'cerebral palsy', 'autism', 'intellectual disability',
        'physical disability', 'accessible', 'inclusive health'
    ],
    'general_srh': [
        'sexual health', 'reproductive health', 'sex education', 'vaginal', 'vulva',
        'penis', 'testicles', 'ovary', 'uterus', 'cervix', 'pelvic',
        'sexual intercourse', 'virginity', 'hymen', 'libido', 'sex drive',
        'masturbation', 'orgasm', 'lubrication', 'genital', 'anatomy'
    ]
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text

def keyword_match_topic(text: str) -> str | None:
    """Return the FIRST matching topic for a text string, or None."""
    text_lower = text.lower()
    for topic, keywords in TOPIC_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            return topic
    return None

print('Setup complete. Topic classes:', list(TOPIC_MAP.keys()))

---
## STEP 1 — Dataset 1: MedMCQA (pre-labeled by medical subject)

**Download link:** https://huggingface.co/datasets/openlifescienceai/medmcqa  
**License:** Apache 2.0  
**Key column:** `subject_name` — we filter for SRHR-relevant subjects

**SRHR subject → our topic mapping:**
```
Obstetrics and Gynecology (O&G)  → pregnancy / contraception / general_srh
Microbiology                     → sti_hiv
Preventive & Social Medicine     → gbv_consent / general_srh
Pediatrics                       → puberty / general_srh
```

**Schema:**
```
id | question | opa | opb | opc | opd | cop | choice_type | exp | subject_name | topic_name
```

In [ ]:
# ── Load MedMCQA ──────────────────────────────────────────────────────────────
print('Loading MedMCQA...')
mq_raw = load_dataset('openlifescienceai/medmcqa')

mq_train = mq_raw['train'].to_pandas()
mq_val   = mq_raw['validation'].to_pandas()
mq_df    = pd.concat([mq_train, mq_val], ignore_index=True)

print(f'Total rows: {len(mq_df)}')
print(f'Columns: {list(mq_df.columns)}')
print(f'\nAll subjects found:')
print(mq_df['subject_name'].value_counts().to_string())

In [ ]:
# ── Map MedMCQA subjects to SRH topics ────────────────────────────────────────

# Subject → topic assignment
# We use keyword matching on the question itself to refine within broad subjects
MEDMCQA_SUBJECT_MAP = {
    'Obstetrics and Gynecology': None,          # will be refined by keyword
    'Microbiology': 'sti_hiv',                  # direct map
    'Preventive & Social Medicine': None,        # will be refined by keyword
    'Pediatrics': None,                          # will be refined by keyword
    'Pharmacology': None,                        # only SRHR drug questions
    'Medicine': None,                            # only SRHR questions
}

# Filter to only SRHR-relevant subjects
mq_srh = mq_df[mq_df['subject_name'].isin(MEDMCQA_SUBJECT_MAP.keys())].copy()
print(f'Rows after subject filter: {len(mq_srh)}')
print(mq_srh['subject_name'].value_counts())

In [ ]:
# ── Assign topics within each subject ─────────────────────────────────────────

def assign_medmcqa_topic(row):
    subject = row['subject_name']
    question = clean_text(str(row['question']))
    
    # Microbiology → always sti_hiv
    if subject == 'Microbiology':
        return 'sti_hiv'
    
    # For all others, try keyword matching first
    kw_topic = keyword_match_topic(question)
    if kw_topic:
        return kw_topic
    
    # Fallback by subject
    if subject == 'Obstetrics and Gynecology':
        return 'pregnancy'      # most OG questions are pregnancy-related
    if subject == 'Preventive & Social Medicine':
        return 'general_srh'
    if subject == 'Pediatrics':
        return 'puberty'
    
    return None   # drop if no match

mq_srh['topic'] = mq_srh.apply(assign_medmcqa_topic, axis=1)
mq_srh = mq_srh[mq_srh['topic'].notna()].copy()

# Use the question text as our training text
mq_srh['text']   = mq_srh['question'].apply(clean_text)
mq_srh['source'] = 'medmcqa'

# Remove short texts
mq_srh = mq_srh[mq_srh['text'].str.len() >= 15].copy()
mq_srh = mq_srh.drop_duplicates(subset='text').reset_index(drop=True)

mq_clean = mq_srh[['text', 'topic', 'source']].copy()

print(f'MedMCQA after cleaning: {len(mq_clean)} rows')
print(mq_clean['topic'].value_counts())

---
## STEP 2 — Dataset 2: AdaptLLM med_knowledge_prob (filename = topic)

**Download link:** https://huggingface.co/datasets/AdaptLLM/med_knowledge_prob  
**License:** Apache 2.0  
**Key insight:** Each specialty is a separate .jsonl file — the filename IS the topic label.

**SRHR file → topic mapping:**
```
Gynaecology & Obstetrics.jsonl    → pregnancy / contraception
Microbiology.jsonl                → sti_hiv
Social & Preventive Medicine.jsonl → gbv_consent / general_srh
Pediatrics.jsonl                  → puberty
```

**Schema:** `input`, `output`, `domain` (specialty name)

In [ ]:
# ── Load AdaptLLM med_knowledge_prob ─────────────────────────────────────────
# This dataset requires loading each specialty as a separate config.
# Available configs: 'Anaesthesia', 'Anatomy', 'Biochemistry', 'Dental', 'ENT',
# 'Forensic Medicine', 'Gynaecology & Obstetrics', 'Medicine', 'Microbiology',
# 'Ophthalmology', 'Orthopedics', 'Pathology', 'Pediatrics', 'Pharmacology',
# 'Physiology', 'Psychiatry', 'Radiology', 'Skin', 'Social & Preventive Medicine',
# 'Surgery', 'Unknown'
# We load ONLY the 4 SRHR-relevant specialties.

print('Loading AdaptLLM med_knowledge_prob (SRHR specialties only)...')

ADAPT_SRHR_CONFIGS = {
    'Gynaecology & Obstetrics':     None,       # keyword refine → pregnancy/contraception
    'Microbiology':                 'sti_hiv',  # direct map
    'Social & Preventive Medicine': None,       # keyword refine → gbv_consent/general_srh
    'Pediatrics':                   'puberty',  # direct map
}

adapt_frames = []
for config_name, default_topic in ADAPT_SRHR_CONFIGS.items():
    print(f'  Loading config: "{config_name}"...')
    try:
        ds = load_dataset('AdaptLLM/med_knowledge_prob', config_name)
        for split in ds.keys():
            df = ds[split].to_pandas()
            df['domain'] = config_name          # tag with specialty name
            df['default_topic'] = default_topic # carry the direct-map topic
            adapt_frames.append(df)
            print(f'    {split}: {len(df)} rows')
    except Exception as e:
        print(f'    WARNING: could not load "{config_name}": {e}')

adapt_df = pd.concat(adapt_frames, ignore_index=True)
print(f'\nTotal AdaptLLM rows loaded: {len(adapt_df)}')
print(f'Columns: {list(adapt_df.columns)}')
print(f'\nRows per specialty:')
print(adapt_df['domain'].value_counts().to_string())
adapt_df.head(2)

In [ ]:
# ── Assign topics and clean AdaptLLM rows ────────────────────────────────────

# Detect text column — AdaptLLM uses 'input' for the question text
text_col = None
for col in ['input', 'question', 'text', 'prompt']:
    if col in adapt_df.columns:
        text_col = col
        break
print(f'Using text column: "{text_col}"')
print(f'All columns: {list(adapt_df.columns)}')

def assign_adapt_topic(row):
    domain        = str(row.get('domain', ''))
    default_topic = row.get('default_topic', None)   # pre-set for Microbiology / Pediatrics
    text          = clean_text(str(row.get(text_col, '')))
    
    # Use the direct map if already set
    if pd.notna(default_topic) and default_topic:
        return default_topic
    
    # For None-mapped specialties, try keyword matching first
    kw = keyword_match_topic(text)
    if kw:
        return kw
    
    # Fallback by specialty
    if domain == 'Gynaecology & Obstetrics':
        return 'pregnancy'
    if domain == 'Social & Preventive Medicine':
        return 'general_srh'
    return None

adapt_df['topic']  = adapt_df.apply(assign_adapt_topic, axis=1)
adapt_srh          = adapt_df[adapt_df['topic'].notna()].copy()
adapt_srh['text']  = adapt_srh[text_col].apply(clean_text)
adapt_srh['source'] = 'adaptllm'

adapt_srh = adapt_srh[adapt_srh['text'].str.len() >= 15].copy()
adapt_srh = adapt_srh.drop_duplicates(subset='text').reset_index(drop=True)

adapt_clean = adapt_srh[['text', 'topic', 'source']].copy()

print(f'\nAdaptLLM after cleaning: {len(adapt_clean)} rows')
print(adapt_clean['topic'].value_counts())

---
## STEP 3 — Dataset 3: HealthCareMagic-100k (keyword filter → auto-label)

**Download link:** https://huggingface.co/datasets/Malikeh1375/medical-question-answering-datasets  
**License:** Open for research use  
**Key column:** `input` (patient question), `output` (doctor answer)  
**Strategy:** Keyword search on `input` to assign topic label

**What HealthCareMagic looks like:**
```
input                                          | output
-----------------------------------------------|------------------------
Hi doctor, I am 17 and my period is late...    | Hello, based on your...
Doctor, I think I may have chlamydia...        | Chlamydia is caused...
```

In [ ]:
# ── Load HealthCareMagic via Malikeh1375 medical-question-answering-datasets ──
print('Loading HealthCareMagic-100k...')

hcm_raw = load_dataset(
    'Malikeh1375/medical-question-answering-datasets',
    'chatdoctor_healthcaremagic'
)

hcm_df = hcm_raw['train'].to_pandas()

print(f'Total rows: {len(hcm_df)}')
print(f'Columns: {list(hcm_df.columns)}')
hcm_df.head(2)

In [ ]:
# ── Identify text column and apply keyword filter ─────────────────────────────

# HealthCareMagic columns are typically: input, output (or instruction, input, output)
hcm_text_col = None
for col in ['input', 'question', 'instruction']:
    if col in hcm_df.columns:
        hcm_text_col = col
        break

print(f'Using text column: {hcm_text_col}')

# Apply keyword matching — this is the auto-labeling step
hcm_df['text']  = hcm_df[hcm_text_col].apply(clean_text)
hcm_df['topic'] = hcm_df['text'].apply(keyword_match_topic)

# Keep only rows that matched a topic
hcm_srh = hcm_df[hcm_df['topic'].notna()].copy()
hcm_srh['source'] = 'healthcaremagic'

# Drop short / empty rows and duplicates
hcm_srh = hcm_srh[hcm_srh['text'].str.len() >= 15].copy()
hcm_srh = hcm_srh.drop_duplicates(subset='text').reset_index(drop=True)

hcm_clean = hcm_srh[['text', 'topic', 'source']].copy()

print(f'HealthCareMagic after SRHR filtering: {len(hcm_clean)} rows')
print(hcm_clean['topic'].value_counts())

---
## STEP 4 — Dataset 4: AfriMedQA v2 (African context — filter by specialty)

**Download link:** https://huggingface.co/datasets/intronhealth/afrimedqa_v2  
**License:** Open for research  
**Key columns:** `specialty`, `question_clean`, `answer_rationale`, `question_type`  
**Value:** Only African-context medical dataset — critical for cultural relevance

**SRHR specialties to filter:**
```
Obstetrics and Gynecology   → pregnancy / contraception
Infectious Disease          → sti_hiv
Preventive & Community Health → general_srh / gbv_consent
Pediatrics                  → puberty
```

**Consumer queries (CQ) are most valuable** — these are real patient questions in African context

In [ ]:
# ── Load AfriMedQA v2 ─────────────────────────────────────────────────────────
print('Loading AfriMedQA v2...')

afri_raw = load_dataset('intronhealth/afrimedqa_v2')

# Combine all available splits
afri_frames = []
for split in afri_raw.keys():
    df = afri_raw[split].to_pandas()
    afri_frames.append(df)

afri_df = pd.concat(afri_frames, ignore_index=True)

print(f'Total rows: {len(afri_df)}')
print(f'Columns: {list(afri_df.columns)}')
print(f'\nSpecialties:')
if 'specialty' in afri_df.columns:
    print(afri_df['specialty'].value_counts().head(20).to_string())
afri_df.head(2)

In [ ]:
# ── Filter AfriMedQA to SRHR specialties ─────────────────────────────────────

AFRI_SPECIALTY_MAP = {
    'Obstetrics and Gynecology':       None,      # keyword refine
    'Infectious Disease':              'sti_hiv',
    'Preventive & Community Health':   None,       # keyword refine
    'Pediatrics':                      'puberty',
    'Family Medicine':                 None,       # keyword only
    'Internal Medicine':               None,       # keyword only
}

# Determine question text column
afri_text_col = None
for col in ['question_clean', 'question', 'text']:
    if col in afri_df.columns:
        afri_text_col = col
        break
print(f'Using text column: {afri_text_col}')

# Filter to SRHR specialties
if 'specialty' in afri_df.columns:
    afri_srh = afri_df[afri_df['specialty'].isin(AFRI_SPECIALTY_MAP.keys())].copy()
else:
    afri_srh = afri_df.copy()  # fallback: keyword filter all

print(f'Rows after specialty filter: {len(afri_srh)}')

def assign_afri_topic(row):
    specialty = row.get('specialty', '') if 'specialty' in row else ''
    text      = clean_text(str(row.get(afri_text_col, '')))
    
    if specialty == 'Infectious Disease':
        return 'sti_hiv'
    if specialty == 'Pediatrics':
        return 'puberty'
    
    kw = keyword_match_topic(text)
    if kw:
        return kw
    
    if specialty == 'Obstetrics and Gynecology':
        return 'pregnancy'
    if specialty == 'Preventive & Community Health':
        return 'general_srh'
    return None

afri_srh['topic']  = afri_srh.apply(assign_afri_topic, axis=1)
afri_srh           = afri_srh[afri_srh['topic'].notna()].copy()
afri_srh['text']   = afri_srh[afri_text_col].apply(clean_text)
afri_srh['source'] = 'afrimedqa'

afri_srh = afri_srh[afri_srh['text'].str.len() >= 15].copy()
afri_srh = afri_srh.drop_duplicates(subset='text').reset_index(drop=True)

afri_clean = afri_srh[['text', 'topic', 'source']].copy()

print(f'AfriMedQA after cleaning: {len(afri_clean)} rows')
print(afri_clean['topic'].value_counts())

---
## STEP 5 — Merge all four datasets

In [ ]:
# ── Concatenate all sources ────────────────────────────────────────────────────
combined = pd.concat(
    [mq_clean, adapt_clean, hcm_clean, afri_clean],
    ignore_index=True
)

print(f'Combined total rows: {len(combined)}')
print(f'\nTopic distribution (before balancing):')
print(combined['topic'].value_counts())
print(f'\nSource breakdown:')
print(combined['source'].value_counts())

In [ ]:
# ── Cross-dataset deduplication ────────────────────────────────────────────────
before = len(combined)
combined = combined.drop_duplicates(subset='text').reset_index(drop=True)
print(f'Removed {before - len(combined)} duplicates across sources')

# Add integer label column
combined['label'] = combined['topic'].map(TOPIC_MAP)

# Verify no unmapped topics
assert combined['label'].isna().sum() == 0, 'Some topics have no integer label!'

print(f'\nFinal combined: {len(combined)} rows')
print(combined[['topic', 'label']].drop_duplicates().sort_values('label'))

---
## STEP 6 — Balance classes

Some topics (pregnancy, sti_hiv) will have many more examples than others (disability_srh, gbv_consent).  
We balance to **400 rows per class = 2,800 total** — enough for multi-class classification.

⚠️ `disability_srh` may have fewer than 400 rows — we oversample it if needed.

In [ ]:
TARGET_PER_CLASS = 400   # Adjust higher if all classes have enough data

balanced_parts = []

for topic, label in TOPIC_MAP.items():
    subset = combined[combined['topic'] == topic]
    available = len(subset)
    
    if available == 0:
        print(f'⚠️  WARNING: No rows found for topic "{topic}" — skipping')
        continue
    
    if available >= TARGET_PER_CLASS:
        # Undersample
        sample = subset.sample(n=TARGET_PER_CLASS, random_state=42)
        print(f'✓  {topic:<20} : {available:>5} available → sampled {TARGET_PER_CLASS}')
    else:
        # Oversample with replacement (only for small classes)
        sample = subset.sample(n=TARGET_PER_CLASS, replace=True, random_state=42)
        print(f'↑  {topic:<20} : {available:>5} available → oversampled to {TARGET_PER_CLASS}')
    
    balanced_parts.append(sample)

balanced = pd.concat(balanced_parts, ignore_index=True)
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f'\nBalanced dataset: {len(balanced)} rows')
print(balanced['topic'].value_counts())

---
## STEP 7 — Final cleaning pass

In [ ]:
# ── Length filter ─────────────────────────────────────────────────────────────
before = len(balanced)
balanced = balanced[
    (balanced['text'].str.len() >= 15) &
    (balanced['text'].str.len() <= 1000)
].reset_index(drop=True)
print(f'Length filter removed {before - len(balanced)} rows')

# ── Null check ────────────────────────────────────────────────────────────────
print(f'\nNull values:')
print(balanced.isnull().sum())
balanced = balanced.dropna(subset=['text', 'label', 'topic']).reset_index(drop=True)

# ── Add text length column for inspection ────────────────────────────────────
balanced['text_len'] = balanced['text'].str.len()
print(f'\nText length stats:')
print(balanced['text_len'].describe().round(1))

print(f'\nFinal dataset: {len(balanced)} rows, {balanced["topic"].nunique()} topics')

---
## STEP 8 — Visualise dataset composition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Topic Classifier Dataset — Composition', fontsize=13, fontweight='bold')

# Topic distribution
topic_counts = balanced['topic'].value_counts()
colors = ['#3B82F6', '#10B981', '#F59E0B', '#8B5CF6', '#EF4444', '#6366F1', '#EC4899']
topic_counts.plot(kind='barh', ax=axes[0], color=colors[:len(topic_counts)])
axes[0].set_title('Topic Distribution (balanced)')
axes[0].set_xlabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_width())}',
                     (p.get_width(), p.get_y() + p.get_height() / 2),
                     ha='left', va='center', fontsize=9)

# Source breakdown per topic (stacked bar)
pivot = balanced.groupby(['topic', 'source']).size().unstack(fill_value=0)
pivot.plot(kind='barh', stacked=True, ax=axes[1], colormap='tab10')
axes[1].set_title('Source Mix per Topic')
axes[1].set_xlabel('Count')
axes[1].legend(loc='lower right', fontsize=8)

# Text length by topic (box plot)
topic_order = balanced['topic'].unique()
data_for_box = [balanced[balanced['topic'] == t]['text_len'].values for t in topic_order]
axes[2].boxplot(data_for_box, labels=topic_order, vert=False)
axes[2].set_title('Text Length by Topic')
axes[2].set_xlabel('Characters')
axes[2].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('data/topic_dataset_composition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to data/topic_dataset_composition.png')

---
## STEP 9 — Train / Validation / Test split and Save

In [ ]:
final = balanced[['text', 'topic', 'label', 'source']].copy()

# 70% train / 15% val / 15% test — stratified on label
train_df, temp_df = train_test_split(
    final, test_size=0.30, stratify=final['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42
)

# Save splits
train_df.to_csv('data/topic_train.csv', index=False)
val_df.to_csv('data/topic_val.csv', index=False)
test_df.to_csv('data/topic_test.csv', index=False)
final.to_csv('data/topic_labels_full.csv', index=False)

# Save label map for use in notebook 06
with open('data/topic_label_map.json', 'w') as f:
    json.dump({'topic_to_int': TOPIC_MAP, 'int_to_topic': TOPIC_NAMES}, f, indent=2)

print('=' * 55)
print('TOPIC CLASSIFIER DATASET — COMPLETE')
print('=' * 55)
print(f'  Train : {len(train_df):>5} rows')
print(f'  Val   : {len(val_df):>5} rows')
print(f'  Test  : {len(test_df):>5} rows')
print(f'  Total : {len(final):>5} rows')
print(f'  Topics: {final["topic"].nunique()} classes')
print()
print('Files saved:')
for f in ['data/topic_train.csv', 'data/topic_val.csv',
          'data/topic_test.csv', 'data/topic_labels_full.csv',
          'data/topic_label_map.json']:
    print(f'  {f}')

---
## STEP 10 — Inspect samples per class

In [ ]:
for topic in TOPIC_MAP.keys():
    subset = final[final['topic'] == topic]
    if len(subset) == 0:
        print(f'\n=== {topic.upper()} — NO ROWS ===\n')
        continue
    sample = subset.sample(min(2, len(subset)), random_state=99)
    print(f'\n=== {topic.upper()} (label={TOPIC_MAP[topic]}) — {len(subset)} rows ===')
    for _, row in sample.iterrows():
        print(f'  [{row["source"]}] {row["text"][:130]}...')

---
## Summary

| Step | Action | Output |
|------|--------|--------|
| 1 | MedMCQA — filter by `subject_name`, assign topic | `mq_clean` |
| 2 | AdaptLLM — filter by `domain`/specialty file, assign topic | `adapt_clean` |
| 3 | HealthCareMagic — keyword filter on `input`, auto-label | `hcm_clean` |
| 4 | AfriMedQA v2 — filter by `specialty`, keyword refine | `afri_clean` |
| 5 | Merge all four sources | `combined` |
| 6 | Balance: 400 rows/class × 7 classes = 2,800 total | `balanced` |
| 7 | Final cleaning: length filter, null check | |
| 8 | Visualise composition | PNG saved |
| 9 | 70/15/15 stratified split + save CSVs | 5 files saved |

### Dataset download links
- MedMCQA: https://huggingface.co/datasets/openlifescienceai/medmcqa
- AdaptLLM med_knowledge_prob: https://huggingface.co/datasets/AdaptLLM/med_knowledge_prob
- HealthCareMagic: https://huggingface.co/datasets/Malikeh1375/medical-question-answering-datasets
- AfriMedQA v2: https://huggingface.co/datasets/intronhealth/afrimedqa_v2

### License notes
- MedMCQA: Apache 2.0 ✓
- AdaptLLM med_knowledge_prob: Apache 2.0 ✓
- HealthCareMagic: Open for research use ✓
- AfriMedQA v2: Open for research use ✓

### Next step
Feed `data/topic_train.csv` into `06_train_topic_classifier.ipynb`.